# Creating Initial Base Objects

In [0]:
%sql
-- DROP SCHEMA IF EXISTS project.volumes CASCADE;
-- DROP SCHEMA IF EXISTS project.default CASCADE;
-- DROP CATALOG IF EXISTS project CASCADE;


### External Location For Managed Catalog


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS ext_catalog_location
  URL 'abfss://catalog-managed@databrickseteprostorage.dfs.core.windows.net/catalog'
  WITH (STORAGE CREDENTIAL adls_credential)

### External Location for the Source

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS ext_source_location
  URL 'abfss://source@databrickseteprostorage.dfs.core.windows.net'
  WITH (STORAGE CREDENTIAL adls_credential)

### Creating External Volume fover Source for Pipeline triggered By File Arrival

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS src_volume
LOCATION 'abfss://source@databrickseteprostorage.dfs.core.windows.net'

### Creating Managed Catalog

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS project
  MANAGED LOCATION  'abfss://catalog-managed@databrickseteprostorage.dfs.core.windows.net/catalog'

In [0]:
%sql
USE CATALOG project;

### Creating Managed Volume over External Source Location for Unity Catalog Governance

In [0]:
# %sql
# CREATE EXTERNAL VOLUME IF NOT EXISTS volumes.source_regions_volume
#   LOCATION 'abfss://source@databrickseteprostorage.dfs.core.windows.net/regions'

In [0]:
dbutils.widgets.text('file_name','')
dbutils.widgets.text('layer_name','')

In [0]:
file_name = dbutils.widgets.get('file_name')
layer_name = dbutils.widgets.get('layer_name')


### Creating Schema - Bronze, Silver, Gold

In [0]:
create_schema = f"CREATE SCHEMA IF NOT EXISTS {layer_name}"
spark.sql(create_schema)

In [0]:
# catalog_name = "project"
# schema_name = "_ops_volumes"
# volume_source_name = f"source_{file_name}_volume";
# volume_destination_name = f"{layer_name}_{file_name}_volume";
volume_name = f"{file_name}_volume";
source_path = f"abfss://source@databrickseteprostorage.dfs.core.windows.net/{file_name}"
volume_path = f"abfss://{layer_name}@databrickseteprostorage.dfs.core.windows.net/{file_name}"



### Creating External Volumes Over all Source folders

In [0]:
# create_ext_source_locations = f"""CREATE EXTERNAL LOCATION IF NOT EXISTS ext_{file_name}_location
#   URL {source_path}
#   WITH (STORAGE CREDENTIAL adls_credential)"""
# spark.sql(create_ext_source_locations)


In [0]:
catalog_name = "project"
schema_name = "source_volumes"
source_path = f"abfss://source@databrickseteprostorage.dfs.core.windows.net/{file_name}"
volume_source_name = f"src_{file_name}";

spark.sql(f'CREATE SCHEMA IF NOT EXISTS project.{schema_name}')
source_volume = f"""CREATE EXTERNAL VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_source_name}
  LOCATION '{source_path}'""";
spark.sql(source_volume)


### Managed Volume for Checkpoins & Schem Evolution

In [0]:
schema_ops_name = "_ops"
ops_name = 'autoloader'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS project.{schema_ops_name}')
managed_vol = f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_ops_name}.{ops_name}";
spark.sql(managed_vol)

### Managed Volume for Incremental Data Dump

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS ext_bronze_location
URL 'abfss://bronze@databrickseteprostorage.dfs.core.windows.net'
WITH (STORAGE CREDENTIAL adls_credential)

In [0]:
bronze_location = f'abfss://bronze@databrickseteprostorage.dfs.core.windows.net/{file_name}'
bronze_vol = f'''CREATE EXTERNAL VOLUME IF NOT EXISTS project.bronze.raw_{file_name} LOCATION '{bronze_location}' '''
spark.sql(bronze_vol)

In [0]:

# source_volume = f"""CREATE EXTERNAL VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_source_name}
#   LOCATION '{source_path}'""";

# dest_Volume = f"""CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}""";

In [0]:
# # spark.sql(source_volume)
# spark.sql(dest_Volume)

# spark.sql(create_layer_locations) 